# 02 Data Preparation, EDA and Popularity Baseline
Week 2 deliverables: cleaned data, sparsity analysis, most active users and popular items, and a popularity baseline with Top-N output.

**Pipeline:** raw transactions -> collapse same-day repeats -> sensitivity analysis -> leakage-safe filter/sample -> cleaned interactions -> time-based split -> interaction matrix -> popularity baseline.

Every filtering choice is controlled in `src/config.py` and justified from the tables below; change a value there and re-run.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config, data_audit as da, prepare as prep, evaluation as ev

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
config.IMAGES_DIR.mkdir(exist_ok=True)

def save_fig(name):
    plt.tight_layout()
    plt.savefig(config.IMAGES_DIR / name, dpi=150)
    plt.show()

## 1. Load and collapse same-day repeats

In [ ]:
customers = da.load_customers()
articles = da.load_articles()
articles_by_id = articles.set_index("article_id")
raw = da.load_transactions(customer_ids=customers["customer_id"])
tx = prep.collapse_same_day(raw)
print(f"rows before: {len(raw):,} | interactions after collapsing same customer+article+day: {len(tx):,} "
      f"({1 - len(tx) / len(raw):.2%} of rows merged)")
display(tx["n_units"].value_counts().sort_index().head(8).rename("interactions").to_frame())
del raw

**Why collapse.** Week 1 found 9.36% exact duplicate rows. Most are the same item bought several times on one day (typically different sizes, which the data does not record). That is one *preference* expressed several times, so it is treated as one interaction with `n_units` kept as an optional weight. Repeats on *different* days are kept, because they show real re-purchase behaviour.

## 2. Sensitivity analysis: how filters change the dataset
Each row is the dataset that would result from a time window, a minimum number of *distinct items per user* and a minimum number of *distinct buyers per item*, both measured on the **training period only** (the last validation and test days are excluded from the counts), with the same seeded user sample the final pipeline uses. `interactions` counts all periods of the surviving users and items.

In [ ]:
windows = {"last 3 months": 91, "last 6 months": 182, "last 12 months": 365, "all history": None}
sens = prep.sensitivity_table(tx, windows, min_users=(1, 5, 10), min_items=(1, 20, 50),
                              sample_users=config.SAMPLE_USERS, seed=config.RANDOM_SEED)
display(sens)

In [ ]:
pivot = sens.pivot_table(index=["window", "min_user"], columns="min_item", values="interactions_per_user")
plt.figure(figsize=(7, 4.5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="Blues")
plt.title("Interactions per user after filtering")
save_fig("sensitivity_interactions_per_user.png")

**How to choose.** A good working set (a) keeps enough history per user to learn from and still leave something to test on, (b) keeps items with enough evidence, (c) stays small enough to iterate quickly, and (d) does not distort popularity patterns by cutting too aggressively. Fashion is seasonal, so a window shorter than a year hides the seasonal cycle. Write your own justification in section 12 after reading this table.

## 3. Apply the chosen filters (leakage-safe)
Thresholds are measured on the training period only. Users and items that pass form a fixed *universe*; all their window interactions are kept, including validation and test ones. Validation/test purchases by universe users of items outside the universe (new or rarely bought items) are kept separately as `cold_items` for a dedicated cold-start analysis. Users with no training history are excluded from the main comparison and need a popularity fallback in a live system.

In [ ]:
spec = prep.PrepSpec()   # values come from src/config.py
print(spec)
prepared = prep.prepare_working_set(tx, spec)
display(prepared.report)
prep.save_prepared(prepared)
cleaned, cold_items = prepared.cleaned, prepared.cold_items
print("Saved to", config.CLEANED_DIR, "| validation starts", prepared.meta["val_start"], "| test starts", prepared.meta["test_start"])

## 4. Sparsity of the cleaned interaction matrix

In [ ]:
users, items = prep.make_index(cleaned)
matrix = prep.build_interaction_matrix(cleaned, users, items, weighting="binary")
n_u, n_i = matrix.shape
print(f"users {n_u:,} | items {n_i:,} | non-empty cells {matrix.nnz:,}")
print(f"sparsity {prep.matrix_sparsity(matrix):.4%} | density {matrix.nnz / (n_u * n_i):.4%}")
per_user, per_item = matrix.getnnz(axis=1), matrix.getnnz(axis=0)
display(pd.DataFrame({"per user": pd.Series(per_user).describe(), "per item": pd.Series(per_item).describe()}))
print("Gini of item popularity:", round(da.gini(pd.Series(per_item)), 4))
display(da.concentration_table(pd.Series(per_item)).rename("share of interactions").to_frame())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, vals, title in zip(axes, [per_user, per_item], ["Items per user", "Users per item"]):
    sns.histplot(vals, bins=50, log_scale=(True, True), ax=ax); ax.set_title(title)
save_fig("cleaned_activity_distributions.png")

## 5. Most active users and most popular items

In [ ]:
top_users = pd.Series(per_user, index=users).sort_values(ascending=False).head(10).rename("distinct items bought")
display(top_users.to_frame())

pop = pd.Series(per_item, index=items).sort_values(ascending=False).head(15).rename("buyers")
meta = articles_by_id.loc[pop.index, ["prod_name", "product_code", "product_type_name", "colour_group_name"]]
top_items = pop.to_frame().join(meta)
display(top_items)
print("Distinct product_code among the top 15 articles:", top_items["product_code"].nunique())

If several top articles share a `product_code`, they are colour variants of one design. A raw popularity list then repeats the same product, which hurts diversity. This is evidence for de-duplicating Top-N lists by product in later weeks.

## 6. Does popularity drift, and how fast does the catalogue turn over?

In [ ]:
last_days = cleaned[cleaned["t_dat"] >= cleaned["t_dat"].max() - pd.Timedelta(days=13)]
top_recent = set(last_days["article_id"].value_counts().head(100).index)
top_overall = set(cleaned["article_id"].value_counts().head(100).index)
print("Overlap of top-100 items (last 14 days vs whole window):", len(top_recent & top_overall), "of 100")

# Catalogue turnover over ALL customers (before filtering): purchases of items not sold earlier in the window
val_start, test_start = pd.Timestamp(prepared.meta["val_start"]), pd.Timestamp(prepared.meta["test_start"])
window_tx = tx[tx["t_dat"] >= tx["t_dat"].max() - pd.Timedelta(days=(config.WINDOW_DAYS or 10**6) - 1)]
sold_before = set(window_tx.loc[window_tx["t_dat"] < val_start, "article_id"])
for name, part in (("validation", window_tx[(window_tx["t_dat"] >= val_start) & (window_tx["t_dat"] < test_start)]),
                   ("test", window_tx[window_tx["t_dat"] >= test_start])):
    print(f"{name}: {(~part['article_id'].isin(sold_before)).mean():.2%} of all purchases are of items not sold earlier in the window")

A low overlap means popularity changes quickly, which is the reason to compare recency-weighted popularity with plain popularity. The turnover figures show how much of the catalogue is new at evaluation time, which limits what any collaborative model can recommend and motivates content features for new items.

## 7. Time-based split

In [ ]:
split = prep.time_split(cleaned, val_start, test_start)
print("validation starts", split.val_start.date(), "| test starts", split.test_start.date())
display(prep.split_report(split, cold_items))

A random split would leak the future into training, so the data is split by time: train, then validation, then test. Models are tuned on validation only; the test block is touched once, in Week 4. Because users and items were chosen from the training period, every evaluated user and item has training history by construction; the cold-item interactions that this excludes are counted in the last column so the limitation is visible.

## 8. Popularity baseline: global and recency-weighted

In [ ]:
train_m = prep.build_interaction_matrix(split.train, users, items, "binary")
val_m = prep.build_interaction_matrix(split.val, users, items, "binary")
eval_users = np.flatnonzero(val_m.getnnz(axis=1) > 0)
item_share = train_m.getnnz(axis=0) / train_m.shape[0]
print(f"{len(eval_users):,} validation users with at least one purchase; "
      f"{(train_m.getnnz(axis=1)[eval_users] > 0).mean():.2%} have train history")

variants = {"global popularity": None, **{f"recency half-life {h}d": h for h in config.HALF_LIVES_DAYS}}
scores = {name: prep.popularity_scores(split.train, items, h) for name, h in variants.items()}
recs = {name: prep.recommend_popular(s, train_m, eval_users, config.TOP_K, exclude_seen=True)
        for name, s in scores.items()}
results = pd.DataFrame({name: ev.evaluate_topk(r, val_m, eval_users, item_share, len(items))
                        for name, r in recs.items()}).T
display(results)
best = results[f"NDCG@{config.TOP_K}"].idxmax()
print("Best baseline on validation (NDCG):", best)

**Reading this table.** Popularity gives almost every user the same list (only items they already bought are removed), so coverage is tiny and novelty low. That is the expected weakness of the baseline, and the personalised models in Weeks 3 and 4 are judged against these numbers. Exclusion of already-bought items is a design choice: fashion has repeat purchases, so Week 4 should report results with and without it.

## 9. Example Top-N lists

In [ ]:
rng = np.random.default_rng(config.RANDOM_SEED)
sample_users = rng.choice(eval_users, 5, replace=False)
sample_recs = prep.recommend_popular(scores[best], train_m, sample_users, config.TOP_K)
name_of = lambda idx: articles_by_id.loc[items[idx], "prod_name"].tolist()
rows = []
for u, r in zip(sample_users, sample_recs):
    bought = split.train[split.train["customer_id"] == users[u]].tail(3)["article_id"]
    rows.append({"user": str(users[u])[:8], "recent purchases": "; ".join(articles_by_id.loc[bought, "prod_name"]),
                 "popularity recommends": "; ".join(name_of(r[:5]))})
display(pd.DataFrame(rows))

## 10. Outfit-like co-purchase at product-type level (size-controlled)
Week 1 showed that lift at the coarse `product_group_name` level is close to 1 and is distorted by very large baskets. This repeats the analysis for small baskets and the finer `product_type_name`, ignoring pairs seen in few baskets.

In [ ]:
lo, hi = config.LIFT_BASKET_SIZES
window_start = tx["t_dat"].max() - pd.Timedelta(days=(config.WINDOW_DAYS or 10**6) - 1)
tx_win = tx[tx["t_dat"] >= window_start]
baskets, cooc = da.basket_analysis(tx_win, articles, "product_type_name", min_size=lo, max_size=hi)
n_used = int(((baskets["basket_size"] >= lo) & (baskets["basket_size"] <= hi)).sum())
print(f"{n_used:,} baskets with {lo} to {hi} items out of {len(baskets):,} customer-day baskets")
lift = da.lift_matrix(cooc, n_used, min_support=config.LIFT_MIN_SUPPORT)
support = pd.Series(np.diag(cooc.to_numpy()), index=cooc.index).sort_values(ascending=False)
top_types = support.head(20).index
plt.figure(figsize=(11, 8))
sns.heatmap(lift.loc[top_types, top_types], annot=True, fmt=".1f", center=1, cmap="RdBu_r", vmin=0.3, vmax=3)
plt.title("Co-purchase lift between product types (baskets of 2-6 items)")
save_fig("product_type_lift.png")

In [ ]:
pairs = lift.stack().rename("lift").reset_index()
pairs.columns = ["type_a", "type_b", "lift"]
pairs = pairs[pairs["type_a"] < pairs["type_b"]]
pairs["baskets"] = [cooc.loc[a, b] for a, b in zip(pairs["type_a"], pairs["type_b"])]
display(pairs.sort_values("lift", ascending=False).head(15))
display(pairs.sort_values("lift").head(10))

Lift above 1 means two product types are bought together more than chance. The strongest well-supported pairs are candidate complementary rules for the outfit-aware layer in Week 4; a weak spread of lift around 1 would argue against using this layer.

## 11. Hand-over to Week 3
`data/cleaned/` (interactions, cold-item interactions and `prepared_meta.json`) is what the personalised model uses. Load it with `prep.load_prepared()` and build the shared matrices with `prep.build_experiment(...)`.

## 12. Findings to write up in your own words
1. **Why collapse same-day repeats, and what it changed:** ...
2. **Chosen window, minimum distinct items per user, minimum buyers per item and sample size, with the numbers from the sensitivity table that justify them, and why thresholds are measured on the training period only:** ...
3. **Sparsity of the cleaned matrix and what it means for the models:** ...
4. **Popularity concentration, variant redundancy in top items, and drift over time:** ...
5. **Split design, the cold-item interactions excluded from the main comparison, and how cold start will be handled:** ...
6. **Which baseline variant wins on validation, and its weaknesses (coverage, novelty):** ...
7. **Which product-type pairs show real complementarity, and whether the outfit layer looks worthwhile:** ...